In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Kitchen Waste Monitor</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/gsap/3.12.2/gsap.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;600;700;800&display=swap" rel="stylesheet">
<script src="https://cdn.tailwindcss.com"></script>
<style>
  :root {
    --bg: #0a0e0b;
    --surface: #111714;
    --border: #1e2e22;
    --fresh: #22ff6e;
    --spoiled: #ff3f3f;
    --neutral: #f0f0cc;
    --accent: #b8ff5e;
    --text-dim: #4d6654;
  }

  * { box-sizing: border-box; }
  body {
    font-family: 'Syne', sans-serif;
    background-color: var(--bg);
    color: var(--neutral);
    min-height: 100vh;
    overflow-x: hidden;
  }

  .mono { font-family: 'Space Mono', monospace; }

  /* Grid background */
  body::before {
    content: '';
    position: fixed;
    inset: 0;
    background-image:
      linear-gradient(rgba(34,255,110,0.03) 1px, transparent 1px),
      linear-gradient(90deg, rgba(34,255,110,0.03) 1px, transparent 1px);
    background-size: 40px 40px;
    pointer-events: none;
    z-index: 0;
  }

  .camera-frame {
    position: relative;
    background: #050907;
    border: 1px solid var(--border);
    overflow: hidden;
  }

  .camera-frame::before,
  .camera-frame::after {
    content: '';
    position: absolute;
    width: 20px;
    height: 20px;
    z-index: 10;
  }
  .camera-frame::before {
    top: 8px; left: 8px;
    border-top: 2px solid var(--fresh);
    border-left: 2px solid var(--fresh);
  }
  .camera-frame::after {
    bottom: 8px; right: 8px;
    border-bottom: 2px solid var(--fresh);
    border-right: 2px solid var(--fresh);
  }

  .corner-tr {
    position: absolute; top: 8px; right: 8px;
    width: 20px; height: 20px;
    border-top: 2px solid var(--fresh);
    border-right: 2px solid var(--fresh);
    z-index: 10;
  }
  .corner-bl {
    position: absolute; bottom: 8px; left: 8px;
    width: 20px; height: 20px;
    border-bottom: 2px solid var(--fresh);
    border-left: 2px solid var(--fresh);
    z-index: 10;
  }

  /* Scan line animation */
  .scan-line {
    position: absolute;
    left: 0; right: 0;
    height: 2px;
    background: linear-gradient(90deg, transparent, var(--fresh), transparent);
    animation: scanDown 3s linear infinite;
    z-index: 5;
    opacity: 0.6;
  }

  @keyframes scanDown {
    0% { top: 0%; opacity: 0; }
    5% { opacity: 0.6; }
    95% { opacity: 0.6; }
    100% { top: 100%; opacity: 0; }
  }

  .video-feed {
    width: 100%;
    aspect-ratio: 4/3;
    display: flex;
    align-items: center;
    justify-content: center;
    position: relative;
  }

  /* Pulse ring */
  .pulse-ring {
    width: 80px; height: 80px;
    border-radius: 50%;
    border: 1px solid var(--fresh);
    position: absolute;
    animation: pulseRing 2s ease-out infinite;
    opacity: 0;
  }
  .pulse-ring:nth-child(2) { animation-delay: 0.6s; }
  .pulse-ring:nth-child(3) { animation-delay: 1.2s; }

  @keyframes pulseRing {
    0% { transform: scale(0.5); opacity: 0.8; }
    100% { transform: scale(3); opacity: 0; }
  }

  .status-badge {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    padding: 4px 12px;
    border-radius: 2px;
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 0.12em;
    text-transform: uppercase;
  }

  .status-fresh {
    background: rgba(34,255,110,0.1);
    border: 1px solid rgba(34,255,110,0.4);
    color: var(--fresh);
  }

  .status-spoiled {
    background: rgba(255,63,63,0.1);
    border: 1px solid rgba(255,63,63,0.4);
    color: var(--spoiled);
  }

  .status-scanning {
    background: rgba(240,240,204,0.06);
    border: 1px solid rgba(240,240,204,0.2);
    color: var(--neutral);
  }

  .confidence-bar {
    height: 4px;
    background: var(--border);
    border-radius: 2px;
    overflow: hidden;
    position: relative;
  }

  .confidence-fill {
    height: 100%;
    border-radius: 2px;
    transition: width 0.6s cubic-bezier(0.4,0,0.2,1);
    position: relative;
    overflow: hidden;
  }

  .confidence-fill::after {
    content: '';
    position: absolute;
    top: 0; right: 0; bottom: 0;
    width: 30px;
    background: linear-gradient(90deg, transparent, rgba(255,255,255,0.4));
    animation: shimmer 1.5s ease-in-out infinite;
  }

  @keyframes shimmer {
    0%, 100% { opacity: 0; }
    50% { opacity: 1; }
  }

  .data-row {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 10px 0;
    border-bottom: 1px solid var(--border);
  }
  .data-row:last-child { border-bottom: none; }

  .label {
    font-size: 11px;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    color: var(--text-dim);
    font-family: 'Space Mono', monospace;
  }

  .value {
    font-size: 13px;
    font-weight: 600;
    letter-spacing: 0.03em;
  }

  .log-entry {
    display: flex;
    align-items: flex-start;
    gap: 10px;
    padding: 10px 0;
    border-bottom: 1px solid rgba(30,46,34,0.5);
    animation: fadeSlideIn 0.4s ease;
  }

  @keyframes fadeSlideIn {
    from { opacity: 0; transform: translateY(-8px); }
    to { opacity: 1; transform: translateY(0); }
  }

  .log-time {
    font-size: 10px;
    color: var(--text-dim);
    font-family: 'Space Mono', monospace;
    white-space: nowrap;
    padding-top: 2px;
  }

  .dot {
    width: 7px; height: 7px;
    border-radius: 50%;
    margin-top: 4px;
    flex-shrink: 0;
  }

  .dot-fresh { background: var(--fresh); box-shadow: 0 0 6px var(--fresh); }
  .dot-spoiled { background: var(--spoiled); box-shadow: 0 0 6px var(--spoiled); }
  .dot-scan { background: var(--text-dim); }

  .stat-card {
    background: var(--surface);
    border: 1px solid var(--border);
    padding: 16px;
    position: relative;
    overflow: hidden;
  }

  .stat-card::before {
    content: '';
    position: absolute;
    bottom: 0; left: 0; right: 0;
    height: 2px;
    background: linear-gradient(90deg, var(--fresh), transparent);
    opacity: 0.4;
  }

  .glow-fresh { text-shadow: 0 0 20px rgba(34,255,110,0.5); }
  .glow-spoiled { text-shadow: 0 0 20px rgba(255,63,63,0.5); }

  .blink { animation: blink 1.2s step-end infinite; }
  @keyframes blink { 50% { opacity: 0; } }

  .btn {
    padding: 10px 20px;
    font-family: 'Space Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    border: 1px solid;
    cursor: pointer;
    transition: all 0.2s;
    background: transparent;
  }

  .btn-primary {
    border-color: var(--fresh);
    color: var(--fresh);
  }
  .btn-primary:hover {
    background: var(--fresh);
    color: var(--bg);
  }

  .btn-danger {
    border-color: var(--spoiled);
    color: var(--spoiled);
  }
  .btn-danger:hover {
    background: var(--spoiled);
    color: white;
  }

  /* Class distribution bars */
  .dist-bar-container {
    display: flex;
    align-items: center;
    gap: 8px;
    margin: 6px 0;
  }
  .dist-bar-label {
    width: 80px;
    font-size: 11px;
    font-family: 'Space Mono', monospace;
    color: var(--text-dim);
    text-transform: uppercase;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
  }
  .dist-bar-track {
    flex: 1;
    height: 6px;
    background: var(--border);
    border-radius: 1px;
  }
  .dist-bar-fill {
    height: 100%;
    border-radius: 1px;
    transition: width 0.8s ease;
  }
  .dist-bar-count {
    font-size: 10px;
    font-family: 'Space Mono', monospace;
    color: var(--text-dim);
    width: 28px;
    text-align: right;
  }

  /* Noise overlay */
  .noise {
    position: fixed;
    inset: 0;
    pointer-events: none;
    z-index: 100;
    opacity: 0.025;
    background-image: url("data:image/svg+xml,%3Csvg viewBox='0 0 256 256' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='noise'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.9' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23noise)'/%3E%3C/svg%3E");
  }

  /* Responsive */
  @media (max-width: 768px) {
    .main-grid { flex-direction: column; }
  }
</style>
</head>
<body>

<div class="noise"></div>

<!-- Header -->
<header class="relative z-10 border-b" style="border-color: var(--border);">
  <div class="max-w-7xl mx-auto px-6 py-4 flex items-center justify-between">
    <div class="flex items-center gap-4">
      <div class="relative">
        <div class="w-8 h-8 flex items-center justify-center" style="border: 1px solid var(--fresh);">
          <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" style="color: var(--fresh);">
            <path d="M2 12s3-7 10-7 10 7 10 7-3 7-10 7-10-7-10-7z"/>
            <circle cx="12" cy="12" r="3"/>
          </svg>
        </div>
        <span class="absolute -top-1 -right-1 w-2 h-2 rounded-full blink" style="background: var(--fresh);"></span>
      </div>
      <div>
        <div class="text-sm font-bold tracking-widest uppercase" style="letter-spacing: 0.2em;">Kitchen Waste Monitor</div>
        <div class="mono text-xs" style="color: var(--text-dim);">v2.1.0 — AI Food Detection System</div>
      </div>
    </div>
    <div class="flex items-center gap-6">
      <div class="mono text-xs" style="color: var(--text-dim);" id="clock">00:00:00</div>
      <div class="flex items-center gap-2">
        <span class="w-2 h-2 rounded-full blink" style="background: var(--fresh);"></span>
        <span class="mono text-xs" style="color: var(--fresh);">LIVE</span>
      </div>
    </div>
  </div>
</header>

<!-- Main content -->
<main class="max-w-7xl mx-auto px-6 py-6 relative z-10">
  <div class="flex gap-6" style="align-items: flex-start;">

    <!-- LEFT: Camera + Controls -->
    <div class="flex flex-col gap-4" style="flex: 1.4;">

      <!-- Camera Feed -->
      <div class="camera-frame" style="border-radius: 0;">
        <div class="corner-tr"></div>
        <div class="corner-bl"></div>
        <div class="scan-line"></div>

        <div class="video-feed" style="background: #030604;">
          <!-- Simulated camera view -->
          <div style="position:relative; width:100%; height:100%; display:flex; align-items:center; justify-content:center;">
            <div class="pulse-ring"></div>
            <div class="pulse-ring"></div>
            <div class="pulse-ring"></div>

            <!-- Simulated food item display -->
            <div id="cameraContent" style="text-align: center;">
              <div style="font-size:64px; margin-bottom: 8px; transition: all 0.4s;" id="foodEmoji">🔍</div>
              <div class="mono text-xs" style="color: var(--text-dim);">SCANNING FOR ITEMS...</div>
            </div>

            <!-- Overlay info (top-left) -->
            <div style="position:absolute; top:12px; left:12px; display:flex; flex-direction:column; gap:4px;" class="mono text-xs" id="overlayInfo">
              <span style="color: var(--text-dim);">CAM_0 // 720p</span>
            </div>

            <!-- Overlay right top -->
            <div style="position:absolute; top:12px; right:12px;">
              <span class="mono text-xs" style="color: var(--text-dim);" id="fpsCounter">30fps</span>
            </div>

            <!-- Detection box (hidden initially) -->
            <div id="detectionBox" style="
              position: absolute;
              width: 140px; height: 140px;
              top: 50%; left: 50%;
              transform: translate(-50%, -50%);
              border: 1.5px solid var(--fresh);
              display: none;
              pointer-events: none;
            ">
              <div style="position:absolute; top:-1px; left:10px; right:10px; height:1px; background: var(--fresh); opacity:0.3;"></div>
              <div style="position:absolute; bottom:-1px; left:10px; right:10px; height:1px; background: var(--fresh); opacity:0.3;"></div>
            </div>
          </div>
        </div>

        <!-- Camera bottom bar -->
        <div style="padding: 8px 14px; border-top: 1px solid var(--border); display: flex; align-items: center; justify-content: space-between; background: rgba(5,9,7,0.8);">
          <div id="statusBadge" class="status-badge status-scanning">
            <span class="blink">●</span> SCANNING
          </div>
          <div class="mono text-xs" style="color: var(--text-dim);" id="intervalLabel">Interval: 500ms</div>
          <div class="flex gap-2">
            <button class="btn btn-primary" onclick="simulateScan()" style="padding:6px 14px; font-size:10px;">
              Scan
            </button>
            <button class="btn btn-danger" onclick="resetDetection()" style="padding:6px 14px; font-size:10px;">
              Clear
            </button>
          </div>
        </div>
      </div>

      <!-- Stats Row -->
      <div class="flex gap-4">
        <div class="stat-card flex-1">
          <div class="label mb-1">Total Scans</div>
          <div class="text-2xl font-bold mono glow-fresh" id="totalScans" style="color: var(--fresh);">0</div>
        </div>
        <div class="stat-card flex-1">
          <div class="label mb-1">Fresh Items</div>
          <div class="text-2xl font-bold mono glow-fresh" id="freshCount" style="color: var(--fresh);">0</div>
        </div>
        <div class="stat-card flex-1">
          <div class="label mb-1">Spoiled Items</div>
          <div class="text-2xl font-bold mono glow-spoiled" id="spoiledCount" style="color: var(--spoiled);">0</div>
        </div>
        <div class="stat-card flex-1" style="border-color: rgba(184,255,94,0.2);">
          <div class="label mb-1">Waste Rate</div>
          <div class="text-2xl font-bold mono" id="wasteRate" style="color: var(--accent);">0%</div>
        </div>
      </div>

    </div>

    <!-- RIGHT: Detection Panel + Log -->
    <div class="flex flex-col gap-4" style="width: 320px; flex-shrink: 0;">

      <!-- Current Detection -->
      <div style="background: var(--surface); border: 1px solid var(--border); padding: 16px;">
        <div class="flex items-center justify-between mb-4">
          <div class="label">Detection Result</div>
          <div class="mono text-xs blink" style="color: var(--text-dim);" id="scanTimer">—</div>
        </div>

        <!-- Item display -->
        <div style="background: #0a0e0b; border: 1px solid var(--border); padding: 16px; margin-bottom: 14px; text-align: center; min-height: 100px; display: flex; flex-direction: column; align-items: center; justify-content: center; gap: 6px;">
          <div style="font-size: 36px;" id="detectionEmoji">—</div>
          <div class="text-base font-bold tracking-wide" id="detectionItem" style="color: var(--neutral);">No Detection</div>
          <div id="detectionStatusBadge" class="status-badge status-scanning" style="font-size:10px;">WAITING</div>
        </div>

        <!-- Data rows -->
        <div>
          <div class="data-row">
            <div class="label">Condition</div>
            <div class="value mono text-xs" id="conditionValue" style="color: var(--text-dim);">—</div>
          </div>
          <div class="data-row">
            <div class="label">Material</div>
            <div class="value mono text-xs" style="color: var(--accent);" id="materialValue">—</div>
          </div>
          <div class="data-row">
            <div class="label">Confidence</div>
            <div class="value mono text-xs" id="confValue" style="color: var(--neutral);">—</div>
          </div>
        </div>

        <!-- Confidence bar -->
        <div class="mt-3">
          <div class="mono text-xs mb-1" style="color: var(--text-dim);">CONFIDENCE LEVEL</div>
          <div class="confidence-bar">
            <div class="confidence-fill" id="confidenceBar" style="width: 0%; background: var(--text-dim);"></div>
          </div>
        </div>
      </div>

      <!-- Class Distribution -->
      <div style="background: var(--surface); border: 1px solid var(--border); padding: 16px;">
        <div class="label mb-3">Detection History</div>
        <div id="distribution">
          <div class="mono text-xs" style="color: var(--text-dim); text-align:center; padding: 16px 0;">No data yet</div>
        </div>
      </div>

      <!-- Activity Log -->
      <div style="background: var(--surface); border: 1px solid var(--border); padding: 16px; flex: 1;">
        <div class="flex items-center justify-between mb-3">
          <div class="label">Activity Log</div>
          <button onclick="clearLog()" class="mono text-xs" style="color: var(--text-dim); background: none; border: none; cursor: pointer; text-decoration: underline;">Clear</button>
        </div>
        <div id="activityLog" style="max-height: 220px; overflow-y: auto;">
          <div class="mono text-xs" style="color: var(--text-dim); text-align: center; padding: 20px 0;">System ready. Awaiting input.</div>
        </div>
      </div>

    </div>
  </div>
</main>

<!-- Footer -->
<footer class="relative z-10 border-t mt-6" style="border-color: var(--border);">
  <div class="max-w-7xl mx-auto px-6 py-3 flex items-center justify-between">
    <div class="mono text-xs" style="color: var(--text-dim);">Kitchen Waste Monitor © 2025 — TensorFlow + OpenCV Backend</div>
    <div class="flex gap-4 mono text-xs" style="color: var(--text-dim);">
      <span>Model: food_waste_model.keras</span>
      <span style="color: var(--border);">|</span>
      <span>Input: 224×224 RGB</span>
      <span style="color: var(--border);">|</span>
      <span>Threshold: 60%</span>
    </div>
  </div>
</footer>

<script>
// ==================== DATA ====================
const items = [
  { name: "Apple", emoji: "🍎", fresh: true, confidence: 94.2 },
  { name: "Banana", emoji: "🍌", fresh: true, confidence: 88.7 },
  { name: "Cucumber", emoji: "🥒", fresh: true, confidence: 91.3 },
  { name: "Apple", emoji: "🍎", fresh: false, confidence: 82.4 },
  { name: "Banana", emoji: "🍌", fresh: false, confidence: 76.8 },
  { name: "Orange", emoji: "🍊", fresh: true, confidence: 95.1 },
  { name: "Tomato", emoji: "🍅", fresh: true, confidence: 89.4 },
  { name: "Tomato", emoji: "🍅", fresh: false, confidence: 73.2 },
  { name: "Mango", emoji: "🥭", fresh: true, confidence: 92.6 },
  { name: "Potato", emoji: "🥔", fresh: false, confidence: 80.1 },
];

let stats = { total: 0, fresh: 0, spoiled: 0 };
let detectionCounts = {};
let logEntries = [];
let scanActive = false;

// ==================== CLOCK ====================
function updateClock() {
  const now = new Date();
  const h = String(now.getHours()).padStart(2,'0');
  const m = String(now.getMinutes()).padStart(2,'0');
  const s = String(now.getSeconds()).padStart(2,'0');
  document.getElementById('clock').textContent = `${h}:${m}:${s}`;
}
setInterval(updateClock, 1000);
updateClock();

// ==================== TIMER ====================
let elapsed = 0;
setInterval(() => {
  elapsed++;
  const m = Math.floor(elapsed/60).toString().padStart(2,'0');
  const s = (elapsed%60).toString().padStart(2,'0');
  document.getElementById('scanTimer').textContent = `${m}:${s}`;
}, 1000);

// ==================== SCAN ====================
function simulateScan() {
  if (scanActive) return;
  scanActive = true;

  // Pick random item
  const item = items[Math.floor(Math.random() * items.length)];
  const isFresh = item.fresh;
  const confidence = (item.confidence + (Math.random() * 6 - 3)).toFixed(1);
  const status = isFresh ? "Not Spoiled (Fresh)" : "Spoiled (Rotten)";
  const className = isFresh ? `Fresh${item.name}` : `Rotten${item.name}`;

  // Show scanning state briefly
  document.getElementById('cameraContent').innerHTML = `
    <div style="font-size:48px; margin-bottom:8px; opacity:0.4;">⏳</div>
    <div class="mono text-xs blink" style="color: var(--fresh);">PROCESSING...</div>
  `;

  setTimeout(() => {
    // Update camera display
    document.getElementById('cameraContent').innerHTML = `
      <div style="font-size:64px; margin-bottom:8px;">${item.emoji}</div>
      <div class="mono text-xs" style="color: ${isFresh ? 'var(--fresh)' : 'var(--spoiled)'};">
        ${isFresh ? '✓ FRESH DETECTED' : '⚠ SPOILED DETECTED'}
      </div>
    `;

    // Show detection box
    const box = document.getElementById('detectionBox');
    box.style.display = 'block';
    box.style.borderColor = isFresh ? 'var(--fresh)' : 'var(--spoiled)';

    // Update status badge (header)
    const badge = document.getElementById('statusBadge');
    badge.className = `status-badge ${isFresh ? 'status-fresh' : 'status-spoiled'}`;
    badge.innerHTML = `<span>●</span> ${isFresh ? 'FRESH' : 'SPOILED'}`;

    // Update detection panel
    document.getElementById('detectionEmoji').textContent = item.emoji;
    document.getElementById('detectionItem').textContent = item.name;

    const detBadge = document.getElementById('detectionStatusBadge');
    detBadge.className = `status-badge ${isFresh ? 'status-fresh' : 'status-spoiled'}`;
    detBadge.textContent = isFresh ? 'FRESH' : 'SPOILED';

    document.getElementById('conditionValue').textContent = status;
    document.getElementById('conditionValue').style.color = isFresh ? 'var(--fresh)' : 'var(--spoiled)';
    document.getElementById('materialValue').textContent = 'Biodegradable';
    document.getElementById('confValue').textContent = `${confidence}%`;

    // Confidence bar
    const bar = document.getElementById('confidenceBar');
    bar.style.width = `${confidence}%`;
    bar.style.background = confidence > 85 ? 'var(--fresh)' : confidence > 70 ? 'var(--accent)' : 'var(--spoiled)';

    // Update stats
    stats.total++;
    if (isFresh) stats.fresh++;
    else stats.spoiled++;

    document.getElementById('totalScans').textContent = stats.total;
    document.getElementById('freshCount').textContent = stats.fresh;
    document.getElementById('spoiledCount').textContent = stats.spoiled;
    document.getElementById('wasteRate').textContent =
      stats.total > 0 ? `${Math.round((stats.spoiled/stats.total)*100)}%` : '0%';

    // Distribution tracking
    const key = `${item.name}_${isFresh ? 'fresh' : 'rotten'}`;
    detectionCounts[key] = (detectionCounts[key] || 0) + 1;
    updateDistribution();

    // Log
    addLog(item.emoji, item.name, isFresh, confidence);

    scanActive = false;
  }, 600);
}

function resetDetection() {
  document.getElementById('cameraContent').innerHTML = `
    <div style="font-size:64px; margin-bottom:8px; transition: all 0.4s;">🔍</div>
    <div class="mono text-xs" style="color: var(--text-dim);">SCANNING FOR ITEMS...</div>
  `;
  document.getElementById('detectionBox').style.display = 'none';
  document.getElementById('statusBadge').className = 'status-badge status-scanning';
  document.getElementById('statusBadge').innerHTML = '<span class="blink">●</span> SCANNING';
  document.getElementById('detectionEmoji').textContent = '—';
  document.getElementById('detectionItem').textContent = 'No Detection';
  document.getElementById('detectionStatusBadge').className = 'status-badge status-scanning';
  document.getElementById('detectionStatusBadge').textContent = 'WAITING';
  document.getElementById('conditionValue').textContent = '—';
  document.getElementById('conditionValue').style.color = 'var(--text-dim)';
  document.getElementById('materialValue').textContent = '—';
  document.getElementById('confValue').textContent = '—';
  document.getElementById('confidenceBar').style.width = '0%';
}

function updateDistribution() {
  const dist = document.getElementById('distribution');
  const total = Object.values(detectionCounts).reduce((a,b)=>a+b,0);
  const sorted = Object.entries(detectionCounts).sort((a,b)=>b[1]-a[1]).slice(0,6);

  dist.innerHTML = sorted.map(([key, count]) => {
    const isFresh = key.endsWith('_fresh');
    const name = key.replace('_fresh','').replace('_rotten','');
    const pct = Math.round((count/total)*100);
    const color = isFresh ? 'var(--fresh)' : 'var(--spoiled)';
    return `
      <div class="dist-bar-container">
        <div class="dist-bar-label">${name}</div>
        <div class="dist-bar-track">
          <div class="dist-bar-fill" style="width:${pct}%; background:${color};"></div>
        </div>
        <div class="dist-bar-count">${count}</div>
      </div>
    `;
  }).join('');
}

function addLog(emoji, name, isFresh, confidence) {
  const now = new Date();
  const time = `${String(now.getHours()).padStart(2,'0')}:${String(now.getMinutes()).padStart(2,'0')}:${String(now.getSeconds()).padStart(2,'0')}`;

  const entry = { emoji, name, isFresh, confidence, time };
  logEntries.unshift(entry);
  if (logEntries.length > 20) logEntries.pop();
  renderLog();
}

function renderLog() {
  const log = document.getElementById('activityLog');
  log.innerHTML = logEntries.map(e => `
    <div class="log-entry">
      <div class="dot ${e.isFresh ? 'dot-fresh' : 'dot-spoiled'}"></div>
      <div style="flex:1;">
        <div style="font-size:12px; font-weight:600;">${e.emoji} ${e.name}
          <span class="mono" style="font-size:10px; color:${e.isFresh ? 'var(--fresh)' : 'var(--spoiled)'}; margin-left:6px;">${e.isFresh ? 'FRESH' : 'ROTTEN'}</span>
        </div>
        <div class="mono" style="font-size:10px; color: var(--text-dim);">Conf: ${e.confidence}% · Biodegradable</div>
      </div>
      <div class="log-time">${e.time}</div>
    </div>
  `).join('');
}

function clearLog() {
  logEntries = [];
  detectionCounts = {};
  stats = { total: 0, fresh: 0, spoiled: 0 };
  document.getElementById('totalScans').textContent = '0';
  document.getElementById('freshCount').textContent = '0';
  document.getElementById('spoiledCount').textContent = '0';
  document.getElementById('wasteRate').textContent = '0%';
  document.getElementById('distribution').innerHTML = '<div class="mono text-xs" style="color: var(--text-dim); text-align:center; padding: 16px 0;">No data yet</div>';
  document.getElementById('activityLog').innerHTML = '<div class="mono text-xs" style="color: var(--text-dim); text-align: center; padding: 20px 0;">Log cleared.</div>';
  resetDetection();
}

// Auto-scan simulation
setInterval(() => {
  simulateScan();
}, 2500);

// Initial scan after 1s
setTimeout(simulateScan, 800);
</script>
</body>
</html>